# 第21章  归一化：把数值拉回安全区间

> BatchNorm/LayerNorm/RMSNorm。Transformer标配LN(每token独立)。LLaMA选RMSNorm(只除RMS不减均值,快15%)。
> 第6章(广播)、第17章(均值/方差)、第20章(浮点)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪')


---
## 21.1-21.5  BN/LN/RMSNorm原理+手写实现

深层网络数值漂移(1.05^100~131)。BN跨batch,序列模型不用。LN每token沿d_model归一化mu=0/std=1。RMSNorm:只除RMS不减均值,LLaMA/Mistral/Qwen标配。

AI连接：第30章Transformer Block中每个Sublayer前有LN/RMSNorm。


In [ ]:
import numpy as np
def ln(x,g=None,b=None,eps=1e-5):
  m=x.mean(axis=-1,keepdims=True); v=x.var(axis=-1,keepdims=True)
  y=(x-m)/np.sqrt(v+eps)
  if g is not None: y=y*g
  if b is not None: y=y+b
  return y
def rn(x,g=None,eps=1e-5):
  r=np.sqrt(np.mean(x**2,axis=-1,keepdims=True))
  y=x/(r+eps)
  if g is not None: y=y*g
  return y
np.random.seed(42); X=np.random.randn(2,3,4)
Xl=ln(X,np.ones(4),np.zeros(4)); Xr=rn(X,np.ones(4))
print(f'LN: mu~0={np.allclose(Xl.mean(axis=-1),0,atol=1e-6)} var~1={np.all(Xl.var(axis=-1)>0.99)}')
print(f'RN: RMS~1={np.allclose(np.sqrt(np.mean(Xr**2,axis=-1)),1,atol=1e-5)}')
print(f'LN mu={Xl.mean():.4f} RN mu={Xr.mean():.4f} (RN keeps original bias)')


---
## 习题

> 在下方新建代码单元格作答。

1.(概念)BN和LN核心区别？Transformer为什么不用BN？
2.(概念)RMSNorm比LayerNorm少了哪一步？为什么可能更好？
3.(代码)NumPy手写LayerNorm(4,8),验证mu=0/var=1,与PyTorch对比。
4.(代码)含极端值向量[1,2,-3,50,0.5]用LN和RMSNorm归一化对比。


---

> 章末钩子：归一化把数值拉回安全区间。但exp(1000)在float32下=Inf——softmax的数值问题。
预览：下一章——**Softmax的死亡与复活**。

> 提示：完成后运行 Kernel -> Restart & Run All 验证。
